In [1]:
# Installation des dépendances (à exécuter une seule fois)
!pip install -U "bitsandbytes>=0.46.1" transformers pandas

import json
import re
import time
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Chargement du modèle Qwen2.5-Math-7B-Instruct (Optimisé 4-bit)
model_name = "Qwen/Qwen2.5-7B-Instruct"
quant_config = BitsAndBytesConfig(load_in_4bit=True)

print("Chargement du tokenizer et du modèle...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)
print("Modèle chargé avec succès !")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 89.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 98.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 33.7 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not current

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modèle chargé avec succès !


In [2]:
# ============================================================
# PROMPTS POUR LE PIPELINE CoVe
# ============================================================

# 1. BASELINE PROMPT : Le modèle donne une première analyse (avec raisonnement libre)
BASELINE_PROMPT = """You are a mathematics expert evaluating whether a proposed list of hypotheses satisfies the requirements of a theorem.
Theorem: {theorem_name}
Proposed hypotheses:
{hypotheses_list}

Analyze the proposed hypotheses against the standard required hypotheses for this theorem. 
Identify if it is an exact match, a valid implication (stronger hypothesis), a missing hypothesis, multiple missing, or an invented hypothesis.
Provide your reasoning."""

# 2. PLANNING PROMPT : Le modèle génère des questions de vérification
PLANNING_PROMPT = """Based on the following theorem, proposed hypotheses, and baseline evaluation, generate 3 specific, independent verification questions to fact-check the evaluation. 
Focus on:
1. What are the exact standard hypotheses required for this theorem?
2. Does a specific proposed hypothesis mathematically imply the required one?
3. Is any required hypothesis completely missing?

Theorem: {theorem_name}
Proposed hypotheses:
{hypotheses_list}
Baseline Evaluation:
{baseline_text}

Return ONLY a numbered list of 3 questions. Do not answer them."""

# 3. EXECUTION PROMPT : Le modèle répond à une question isolée (Factored)
EXECUTION_PROMPT = """You are a mathematical verifier. Answer the following question accurately and concisely based purely on mathematical facts:
Question: {question}"""

# 4. FINAL PROMPT : Le modèle génère le JSON final en prenant en compte les vérifications
FINAL_SYSTEM_PROMPT = """You are a mathematics expert evaluating whether a proposed list of hypotheses correctly satisfies the hypotheses required by a theorem.
You must classify the VALIDATION TYPE into one of: ["correct", "valid_implication", "missing_hypothesis", "multiple_missing", "invented_hypothesis"].

You are provided with a baseline evaluation and a set of verified facts (Q&A). 
Use these verified facts to CORRECT any hallucinations or errors in the baseline evaluation.

OUTPUT FORMAT:
Return ONLY a JSON object:
{
  "label": "correct" or "incorrect",
  "validation_type": "correct" or "valid_implication" or "missing_hypothesis" or "multiple_missing" or "invented_hypothesis",
  "reason": "concise mathematical explanation based on the verified facts"
}
CRITICAL: Return ONLY the JSON object. No markdown. Start with "{" and end with "}"."""

FINAL_USER_PROMPT = """Theorem: {theorem_name}
Proposed hypotheses:
{hypotheses_list}

Baseline Evaluation (may contain errors):
{baseline_text}

Verified Facts (use these to correct the baseline if needed):
{qa_pairs}

Provide the final JSON evaluation."""

# ============================================================
# FONCTION D'APPEL AU LLM
# ============================================================
def call_llm(system_msg, user_msg, max_tokens=500):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# ============================================================
# PIPELINE CoVe COMPLET
# ============================================================
def run_cove_pipeline(theorem_name, hypotheses):
    hyp_text = "\n".join(f"- {h}" for h in hypotheses)
    
    # ÉTAPE 1 : Baseline
    baseline_text = call_llm(
        "You are a helpful math assistant.", 
        BASELINE_PROMPT.format(theorem_name=theorem_name, hypotheses_list=hyp_text)
    )
    
    # ÉTAPE 2 : Planification (Génération des questions)
    planning_text = call_llm(
        "You are an expert at fact-checking mathematical statements.",
        PLANNING_PROMPT.format(theorem_name=theorem_name, hypotheses_list=hyp_text, baseline_text=baseline_text)
    )
    
    # Extraction naïve des questions (basée sur les retours à la ligne)
    questions = [q.strip() for q in planning_text.split('\n') if q.strip() and q[0].isdigit()]
    if not questions:
        questions = [planning_text] # Fallback si le format n'est pas une liste numérotée
        
    # ÉTAPE 3 : Exécution (Factored - Une requête par question)
    qa_pairs = ""
    for q in questions:
        answer = call_llm(
            "You are a strict mathematical verifier.",
            EXECUTION_PROMPT.format(question=q),
            max_tokens=150
        )
        qa_pairs += f"Q: {q}\nA: {answer}\n\n"
        
    # ÉTAPE 4 : Réponse Finale Corrigée
    final_raw = call_llm(
        FINAL_SYSTEM_PROMPT,
        FINAL_USER_PROMPT.format(
            theorem_name=theorem_name, 
            hypotheses_list=hyp_text, 
            baseline_text=baseline_text, 
            qa_pairs=qa_pairs
        ),
        max_tokens=300
    )
    
    return final_raw, baseline_text, qa_pairs

In [9]:
import json

import json
import re

def parse_llm_output(raw_text):
    if not raw_text:
        return {"label": None, "validation_type": None, "reason": None, "parse_error": "no_response"}
    
    cleaned = re.sub(r"```(?:json)?", "", raw_text, flags=re.IGNORECASE).strip()
    
    # CORRECTION : Échapper les antislashs LaTeX générés par le LLM pour rendre le JSON valide
    cleaned = cleaned.replace('\\', '\\\\')
    
    try:
        start = cleaned.find('{')
        end = cleaned.rfind('}') + 1
        if start != -1 and end != -1:
            parsed = json.loads(cleaned[start:end])
            
            val_type = parsed.get("validation_type")
            if val_type == "correct":
                val_type = "exact_match"
                
            return {
                "label": parsed.get("label"),
                "validation_type": val_type,
                "reason": parsed.get("reason"),
                "parse_error": None,
            }
    except Exception as e:
        return {"label": None, "validation_type": None, "reason": None, "parse_error": str(e)}
        
    return {"label": None, "validation_type": None, "reason": None, "parse_error": "JSON decode error"}
# ============================================================
# EXÉCUTION DU BENCHMARK AVEC CoVe
# ============================================================

INPUT_JSON_PATH = "/kaggle/input/datasets/mathurasanthalingam/benchmark-students-copy/benchmark_data.json"
OUTPUT_CSV_PATH = "/kaggle/working/benchmark_cove_results.csv"

with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Pour tester sans attendre des heures, on peut limiter aux 10 premiers exemples : dataset = dataset[:10]
results = []
start_time = time.time()

print(f"Lancement de CoVe sur {len(dataset)} exemples...")

for i, sample in enumerate(dataset):
    theorem_name = sample["name"]
    hypotheses = sample["copy"]
    
    try:
        final_raw, baseline, qa_pairs = run_cove_pipeline(theorem_name, hypotheses)
        parsed = parse_llm_output(final_raw)
    except Exception as e:
        final_raw, baseline, qa_pairs = str(e), "", ""
        parsed = {"label": None, "validation_type": None, "reason": None, "parse_error": str(e)}

    results.append({
        "theorem_id": sample["theorem_id"],
        "name": theorem_name,
        "ground_truth_label": "correct" if sample["is_correct"] else "incorrect",
        "ground_truth_validation": sample["validation_type"],
        
        "prediction_label": parsed["label"],
        "prediction_validation": parsed["validation_type"],
        "prediction_reason": parsed["reason"],
        
        "cove_baseline": baseline,
        "cove_qa_pairs": qa_pairs,
        "parse_error": parsed["parse_error"],
        "raw_response": final_raw,
    })
    
    if (i + 1) % 5 == 0:
        print(f"{i+1}/{len(dataset)} exemples traités...")

# Sauvegarde
df_cove = pd.DataFrame(results)
df_cove.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"\nTerminé ! Sauvegardé dans {OUTPUT_CSV_PATH}")
print(f"Temps d'exécution : {(time.time() - start_time) / 60:.2f} minutes")

# Affichage d'un aperçu
display(df_cove[['name', 'ground_truth_validation', 'prediction_validation', 'parse_error']].head())

Lancement de CoVe sur 10 exemples...
5/10 exemples traités...
10/10 exemples traités...

Terminé ! Sauvegardé dans /kaggle/working/benchmark_cove_results.csv
Temps d'exécution : 14.08 minutes


,name,ground_truth_validation,prediction_validation,parse_error
0,Adjacent Sequences Theorem,exact_match,exact_match,None
1,Rolle's Theorem,multiple_missing,missing_hypothesis,None
2,Cauchy-Lipschitz Theorem,exact_match,exact_match,None
3,Cauchy Criterion (sequences),exact_match,missing_hypothesis,None
4,Intermediate Value Theorem,stronger_hypothesis,valid_implication,None
